# M2 — Moment-Type-Aware Temporal Plateau Solver

Bounded semantic-moment localization inside the exact MB1 candidate raw-video window. A0 is frozen M1 coarse-to-fine, A1 is dense raw-frame CLIP argmax, and A2 routes over a fixed half-prominence plateau. This is not global retrieval and does not integrate T3.

## INPUT CẦN GẮN TRÊN KAGGLE

1. Raw AIC videos: `/kaggle/input/datasets/nadkli/dataset-aic`
2. MB1 candidates: `/kaggle/input/datasets/irthn1311/triage-eg-mb1-candidates`
3. MB1 annotations: `/kaggle/input/datasets/irthn1311/triage-eg-mb1-ai-annotations`
4. Stage 1B verification: `/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports`
5. Offline OpenAI CLIP: `/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32`

Nested roots are resolved with bounded traversal. No Stage1 vectors, Stage1E/OPUS, T3, OCR, ASR, Objects, VLM, or training assets are needed. No model is downloaded.

## OUTPUT ZIP

`/kaggle/working/triage_eg_moment_m2_bundle.zip`


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path("/kaggle/working/AIC2026_TeamPTK_SGU")
if not (REPO_DIR / ".git").is_dir():
    clone_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
        env=clone_env,
    )
sys.path.insert(0, str(REPO_DIR / "src"))
COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print("resolved commit:", COMMIT)
print("Experiment runtime uses offline assets only; no model download.")


In [ ]:
DATASET_INPUT = Path(
    os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic")
)
CANDIDATE_INPUT = Path(
    os.environ.get(
        "AIC_MB1_CANDIDATE_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-mb1-candidates",
    )
)
ANNOTATION_INPUT = Path(
    os.environ.get(
        "AIC_MB1_ANNOTATION_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-mb1-ai-annotations",
    )
)
STAGE1B_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1B_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports",
    )
)
CLIP_INPUT = Path(
    os.environ.get(
        "AIC_OPENAI_CLIP_ASSET_ROOT",
        "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32",
    )
)
OUTPUT_ROOT = Path("/kaggle/working/triage_eg_moment_m2")
ZIP_PATH = Path("/kaggle/working/triage_eg_moment_m2_bundle.zip")
print(
    {
        "raw": str(DATASET_INPUT),
        "candidates": str(CANDIDATE_INPUT),
        "annotations": str(ANNOTATION_INPUT),
        "stage1b": str(STAGE1B_INPUT),
        "clip": str(CLIP_INPUT),
        "output": str(OUTPUT_ROOT),
        "zip": str(ZIP_PATH),
    }
)


In [ ]:
SEARCH_ROOT = Path("/kaggle/input")
MAX_DEPTH = 6
MAX_DIRECTORIES = 5000

def bounded_files(root: Path, filename: str):
    matches, frontier, visited = [], [(Path(root), 0)], 0
    while frontier:
        current, depth = frontier.pop(0)
        if not current.is_dir():
            continue
        visited += 1
        if visited > MAX_DIRECTORIES:
            raise RuntimeError(f"Input discovery exceeded {MAX_DIRECTORIES} directories")
        direct = current / filename
        if direct.is_file():
            matches.append(direct.resolve())
        if depth < MAX_DEPTH:
            frontier.extend(
                (child, depth + 1)
                for child in sorted(current.iterdir())
                if child.is_dir()
            )
    return sorted(set(matches))

def resolve_unique_file(requested: Path, filename: str) -> Path:
    if requested.is_file() and requested.name == filename:
        return requested.resolve()
    matches = bounded_files(requested, filename)
    if not matches:
        matches = bounded_files(SEARCH_ROOT, filename)
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {filename}; found {matches}")
    return matches[0]

def resolve_root(requested: Path, marker: str) -> Path:
    marker_path = resolve_unique_file(requested, Path(marker).name)
    if not marker_path.as_posix().endswith(marker):
        candidates = [
            path
            for path in bounded_files(requested, Path(marker).name)
            if path.as_posix().endswith(marker)
        ]
        if len(candidates) != 1:
            raise RuntimeError(f"Expected one root with {marker}; found {candidates}")
        marker_path = candidates[0]
    root = marker_path
    for _ in Path(marker).parts:
        root = root.parent
    return root.resolve()

def resolve_dataset_root(requested: Path) -> Path:
    probe = resolve_unique_file(requested, "L23_V005.mp4")
    if probe.parent.name != "video":
        raise RuntimeError(f"Unexpected raw-video layout: {probe}")
    return probe.parents[2].resolve()

DATASET_ROOT = resolve_dataset_root(DATASET_INPUT)
CANDIDATE_PATH = resolve_unique_file(CANDIDATE_INPUT, "mb1_candidate_manifest.jsonl")
ANNOTATION_PATH = resolve_unique_file(ANNOTATION_INPUT, "mb1_ai_semantic_moments.jsonl")
STAGE1B_ROOT = resolve_root(STAGE1B_INPUT, "encoder/selected_encoder_contract.json")
CLIP_ROOT = resolve_root(CLIP_INPUT, "checkpoint/ViT-B-32.pt")
print(
    json.dumps(
        {
            "dataset_root": str(DATASET_ROOT),
            "candidate_manifest": str(CANDIDATE_PATH),
            "annotations": str(ANNOTATION_PATH),
            "stage1b_root": str(STAGE1B_ROOT),
            "clip_root": str(CLIP_ROOT),
        },
        indent=2,
    )
)


In [ ]:
from triage_eg.experiments.moment_m2 import M2Config, preflight_m2

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
CONFIG = M2Config(
    dataset_root=DATASET_ROOT,
    candidate_manifest_path=CANDIDATE_PATH,
    annotation_path=ANNOTATION_PATH,
    stage1b_root=STAGE1B_ROOT,
    clip_asset_root=CLIP_ROOT,
    output_root=OUTPUT_ROOT,
    seed=2026,
    device=os.environ.get("AIC_CLIP_DEVICE", "auto"),
    batch_size=int(os.environ.get("AIC_CLIP_BATCH_SIZE", "16")),
    build_git_commit=COMMIT,
)
PREFLIGHT = preflight_m2(CONFIG)
print(json.dumps(PREFLIGHT, indent=2))


In [ ]:
from triage_eg.experiments.moment_m2 import run_m2

RESULT = run_m2(CONFIG)
print(json.dumps(RESULT["summary"], indent=2))


In [ ]:
print("ALL MOMENTS")
print(json.dumps(RESULT["metrics"]["SLICES"]["ALL_MOMENTS"], indent=2))
print("BOUNDARY LIKE")
print(json.dumps(RESULT["metrics"]["SLICES"]["BOUNDARY_LIKE"], indent=2))
print("ACTION VISIBILITY GUARD")
print(json.dumps(RESULT["metrics"]["ROUTING_SAFETY"], indent=2))
print("M2_QUALITY_DECISION = NOT_EVALUATED")


In [ ]:
from IPython.display import Image, display

boundary_types = {
    "FIRST_OCCURRENCE",
    "TRANSITION_ONSET",
    "TRANSITION_OFFSET",
    "CONTACT",
    "SEPARATION",
    "LAST_OCCURRENCE",
}
result_lines = (OUTPUT_ROOT / "moment_results.jsonl").read_text(
    encoding="utf-8"
).splitlines()
rows = [json.loads(line) for line in result_lines if line.strip()]
shown = 0
for row in rows:
    path = OUTPUT_ROOT / "visuals" / f"{row['moment_id']}_ab.jpg"
    if row["moment_type"] in boundary_types and path.is_file():
        display(Image(filename=str(path)))
        shown += 1
        if shown == 4:
            break
print("displayed blinded boundary examples:", shown)


In [ ]:
from triage_eg.experiments.moment_m2 import create_m2_bundle

bundle = create_m2_bundle(OUTPUT_ROOT, ZIP_PATH)
print("M2_IMPLEMENTATION_STATUS = COMPLETE")
print("M2_REAL_STATUS = COMPLETE")
print("M2_DIAGNOSTIC_STATUS = COMPLETE")
print("M2_QUALITY_DECISION = NOT_EVALUATED")
print("DOWNLOAD ZIP:", bundle, "size_bytes=", bundle.stat().st_size)
